# Example workflow for SOM integration into StatMagic HMI

Currently **without** support for
- label correlation
- plotting


### Different txt and dictionary files

- bmu_ids.txt
- cluster_hit_count.txt
- result_som.txt
- RunStats.txt
- cluster_label_counts.txt
- geo_labeled_bmu.txt
- labels_flat.txt
- labels_grouped.txt
- **result_geo.txt** 🚨: these are important for plotting, but can become super-large.
- som.dictionary: looks like a binary file
- cluster.dictionary: looks like a binary file

### Plots

- db_score: Davies-Bouldin Index (related to kMeans clustering)
- bmu_id_mesh: SOM grid
- cluster_hit_count: number of pixels per cluster
- cluster_label_count: number of labels per cluster
- geoplots: depending on number of input layers
- boxplots: depending on number of input layers
- somplots: depending on number of input layers

### Folders
- GeoTIFF: **contains all tif outputs, including**
    - Discretized input layers (I think these are some kind of codebook vector maps), files starting with **“b_”**
    - **q_error**
    - **bmu_id**
    - **cluster**
    - **bmu_bmu_label_count**
    - **bmu_cluster_label_count**

### Reminder
Some of these outputs are only relevant if
- kmeans clustering was performed
- labels were provided
- plotting was activated

## Imports

In [ ]:
import pandas as pd

from beak.hmi_integration.call_som import run_som
from beak.hmi_integration.utils import (
    _get_data_folder,
    _select_data,
    _extract_payload,
    _extract_rows,
    _filter_tif_files,
    _download_cdr_files,
)

## Block 1: Get data folder and JSON file

In [ ]:
# 1. Define main data folder: we use the beak/data folder provided by the MTRI package template
data_folder = _get_data_folder()

# 2. Select JSON and read data: use a collective folder for all JSON files
file_path = data_folder / "cdr" / "json" / "BEAK_LACULI.json"
json_data = pd.read_json(file_path)

# 3. Get CMA info etc.: the model_run_id is used as identifier for the subfolder to download files and save model outputs
cma = _select_data(data=json_data, target="cma")
model_run_id = _extract_payload(data=json_data, target="model_run_id", normalize=False)

# 4. Extract evidence layers
evidence_layers = _extract_payload(data=json_data, target="evidence_layers", normalize=True)

# 5. Extract download url's from evidence layers: get the list of all urls in the payload
download_urls = _extract_rows(data=evidence_layers, attribute="download_url")

# 6. Filter tif files: only keep .tif/.tiff files left for downloading
filtered_urls = _filter_tif_files(file_list=download_urls)

# 7. Download files and create file list to be used as input file list for the SOM call
identifier = model_run_id
download_folder = data_folder / "cdr" / identifier / "inputs"

input_file_list = _download_cdr_files(
    download_urls=filtered_urls,
    download_folder=download_folder,
    verify_ssl=False,
)


## Block 2: Run the SOM algorithm

In [ ]:
# 1. Run SOM algorithm
output_folder = data_folder / "cdr" / identifier / "outputs"

run_som(
  input_layers=input_file_list,
  config_file=file_path,
  output_folder=output_folder
)